# London seasonal Poisson calibration

This notebook estimates the annual transmission amplitude and peak week together with the eight existing London parameters. It minimises the same deterministic, overlap-weighted Poisson negative log-likelihood used by the non-seasonal Poisson fit. The exceptional 2024 wave is excluded: 5 August 2024 is retained only as the first post-wave conditioning observation, and scored rolling targets begin on 12 August 2024.

The earlier sensitivity grid selected its boundaries, $a=0.30$ and $w_{\mathrm{peak}}=13$. Those were not estimates. This fit widens the amplitude range to $[0,0.60]$ and searches the complete annual phase $[0,52.18)$. The annual period remains fixed at $T=52.18$ weeks.

The fit uses all available post-wave weeks, including both later peaks and the intervening troughs. It is a descriptive post-wave calibration, not an independent validation experiment. Do not use its all-data parameter vector to claim held-out forecast performance.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

ROOT = Path.cwd().resolve()
if ROOT.name == 'outbreak_probability_model':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from outbreak_probability_model.model import load_default_inputs
from outbreak_probability_model.london_calibration import (
    CalibrationConfig, DEFAULT_LONDON_POISSON_FITTED_PARAMETERS,
    DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS,
    FIT_PARAMETER_BOUNDS, PIPELINE_VERSION,
    SEASONAL_FIT_PARAMETER_BOUNDS, default_calibration_parameters,
    evaluate_vector, fit_shared_parameters, fitted_parameter_table,
    load_london_confirmed_cases, load_london_fitted_parameters,
    make_six_week_blocks,
)

EXPECTED_PIPELINE_VERSION = 'mathsy-london-v7-random-search-120-40'
if PIPELINE_VERSION != EXPECTED_PIPELINE_VERSION:
    raise RuntimeError(
        f'Stale notebook kernel: loaded {PIPELINE_VERSION}, expected '
        f'{EXPECTED_PIPELINE_VERSION}. Restart the kernel and run all cells.'
    )
OUTPUT = DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS.parent
OUTPUT.mkdir(parents=True, exist_ok=True)
ALL_SERIES_SEASONAL_FITTED_PARAMETERS = (
    ROOT / 'experiments/measles/London/'
    'calibration_6week_rolling_poisson_seasonal_all_series_117_windows/'
    '03_seasonal_poisson_fitted_parameters.csv'
)
QUICK_CHECK = False
RUN_OPTIONAL_GRADIENT = False  # diagnostic only; never replaces the main search fit
SEASONAL_EXCLUSION_END = pd.Timestamp('2024-08-04')
if QUICK_CHECK:
    config = CalibrationConfig(n_trials=20, n_refinement_trials=6,
        initial_state_refinement_maxiter=0, seasonal_refinement_maxiter=4,
        random_seed=20260823)
    GRADIENT_MAXITER = 8
else:
    config = CalibrationConfig(n_trials=120, n_refinement_trials=40,
        initial_state_refinement_maxiter=60, seasonal_refinement_maxiter=80,
        random_seed=20260823)
    GRADIENT_MAXITER = 140
print('Output directory:', OUTPUT)
print('Seasonal bounds:', SEASONAL_FIT_PARAMETER_BOUNDS)
print('Optional gradient diagnostic:', RUN_OPTIONAL_GRADIENT)

## 1. Observations, windows and starting point

The existing non-seasonal Poisson fit supplies a safe warm start for the original eight parameters. The provisional seasonal grid result supplies only the starting coordinates $a=0.30$ and $w_{\mathrm{peak}}=13$; the optimiser is free to move away from both.

In [ ]:
if not DEFAULT_LONDON_POISSON_FITTED_PARAMETERS.exists():
    raise FileNotFoundError(
        'Run London_Calibration_6Week_Rolling_Poisson_Fit.ipynb first: ' +
        str(DEFAULT_LONDON_POISSON_FITTED_PARAMETERS)
    )
all_cases = load_london_confirmed_cases()
# Exclude the exceptional 2024 wave from both scored targets and the
# conditioning anchor. The first retained post-wave observation becomes
# week 0, so scored forecasts begin one week later. Building blocks only
# after filtering also recomputes overlap coverage on the retained sample.
cases = all_cases.loc[all_cases.date > SEASONAL_EXCLUSION_END].reset_index(drop=True)
blocks = make_six_week_blocks(cases, block_weeks=6, step_weeks=1)
inputs = load_default_inputs()
base = default_calibration_parameters()
_, nonseasonal_vector = load_london_fitted_parameters(
    DEFAULT_LONDON_POISSON_FITTED_PARAMETERS
)
_, all_series_seasonal_vector = load_london_fitted_parameters(
    ALL_SERIES_SEASONAL_FITTED_PARAMETERS
)
starting_vector = {name: float(nonseasonal_vector[name]) for name in FIT_PARAMETER_BOUNDS}
starting_vector.update(seasonal_amplitude=0.30, seasonal_peak_week=13.0)
settings = pd.DataFrame([{
    'pipeline_version': PIPELINE_VERSION, 'quick_check': QUICK_CHECK,
    'all_observed_weeks': len(all_cases), 'post_wave_observed_weeks': len(cases),
    'rolling_windows': blocks.block_id.nunique(),
    'excluded_through': SEASONAL_EXCLUSION_END.date(),
    'first_conditioning_week': cases.date.min().date(),
    'first_scored_week': blocks.date.min().date(),
    'last_scored_week': blocks.date.max().date(),
    'objective': 'overlap-weighted Poisson NLL',
    'seasonal_period_weeks': 52.18,
    'amplitude_lower': SEASONAL_FIT_PARAMETER_BOUNDS['seasonal_amplitude'][0],
    'amplitude_upper': SEASONAL_FIT_PARAMETER_BOUNDS['seasonal_amplitude'][1],
    'peak_week_lower': SEASONAL_FIT_PARAMETER_BOUNDS['seasonal_peak_week'][0],
    'peak_week_upper': SEASONAL_FIT_PARAMETER_BOUNDS['seasonal_peak_week'][1],
    'n_trials': config.n_trials, 'n_refinement_trials': config.n_refinement_trials,
    'seasonal_refinement_maxiter': config.seasonal_refinement_maxiter,
    'initial_state_refinement_maxiter': config.initial_state_refinement_maxiter,
    'run_optional_gradient': RUN_OPTIONAL_GRADIENT,
    'gradient_maxiter': GRADIENT_MAXITER,
    'random_seed': config.random_seed,
}])
display(settings)
settings.to_csv(OUTPUT / '00_seasonal_poisson_fit_settings.csv', index=False)

## 2. Joint seasonal Poisson fit

All ten quantities participate in the bounded random search and local random refinement. Bounded Powell searches then refine the seasonal pair and the two hidden-state multipliers. This search-derived vector is always the canonical seasonal fit. Set `RUN_OPTIONAL_GRADIENT=True` only to generate a separate L-BFGS-B diagnostic; it cannot overwrite the main fitted vector.

In [ ]:
search_vector, trials, search_diagnostics = fit_shared_parameters(
    blocks, inputs=inputs, config=config, base_parameters=base, progress=True,
    overlap_weighted=True, objective_metric='poisson_nll',
    starting_vector=starting_vector,
    parameter_bounds=SEASONAL_FIT_PARAMETER_BOUNDS,
)
trials.to_csv(OUTPUT / '01_seasonal_poisson_optimization_trials.csv', index=False)
search_score, search_diagnostics = evaluate_vector(
    search_vector, blocks, inputs, base, overlap_weighted=True,
    objective_metric='poisson_nll',
)

parameter_names = list(SEASONAL_FIT_PARAMETER_BOUNDS)
lower = np.array([SEASONAL_FIT_PARAMETER_BOUNDS[name][0] for name in parameter_names])
upper = np.array([SEASONAL_FIT_PARAMETER_BOUNDS[name][1] for name in parameter_names])
scale = upper - lower
gradient_history = []

def vector_to_unit(vector):
    values = np.array([vector[name] for name in parameter_names], dtype=float)
    return np.clip((values - lower) / scale, 0.0, 1.0)

def unit_to_vector(unit_values):
    values = lower + np.asarray(unit_values, dtype=float) * scale
    return dict(zip(parameter_names, values))

def objective_on_unit_scale(unit_values):
    candidate = unit_to_vector(unit_values)
    score, _ = evaluate_vector(
        candidate, blocks, inputs, base, overlap_weighted=True,
        objective_metric='poisson_nll',
    )
    gradient_history.append({'evaluation': len(gradient_history),
                             'poisson_nll': score, **candidate})
    return score

gradient_result = None
gradient_vector = None
gradient_score = np.nan
if RUN_OPTIONAL_GRADIENT:
    gradient_result = minimize(
        objective_on_unit_scale, vector_to_unit(search_vector),
        method='L-BFGS-B', jac='2-point',
        bounds=[(0.0, 1.0)] * len(parameter_names),
        options={'maxiter': GRADIENT_MAXITER, 'ftol': 1e-10,
                 'gtol': 1e-6, 'maxls': 40},
    )
    gradient_vector = unit_to_vector(gradient_result.x)
    gradient_score, _ = evaluate_vector(
        gradient_vector, blocks, inputs, base, overlap_weighted=True,
        objective_metric='poisson_nll',
    )
    pd.DataFrame(gradient_history).to_csv(
        OUTPUT / '10_seasonal_gradient_optimization_trials.csv', index=False
    )
    pd.DataFrame([{'pipeline_version': PIPELINE_VERSION, **gradient_vector}]).to_csv(
        OUTPUT / '13_seasonal_gradient_fitted_parameters.csv', index=False
    )
    print(f'Optional diagnostic NLL: {search_score:.8f} -> {gradient_score:.8f}')
else:
    for stale_name in ('10_seasonal_gradient_optimization_trials.csv',
                       '13_seasonal_gradient_fitted_parameters.csv'):
        stale_path = OUTPUT / stale_name
        if stale_path.exists():
            stale_path.unlink()
    print('Optional gradient diagnostic skipped; stale diagnostic outputs removed.')
selected_method = 'bounded random search with local/Powell refinement'
seasonal_vector = search_vector
seasonal_score = search_score
diagnostics = search_diagnostics
pd.DataFrame([{'pipeline_version': PIPELINE_VERSION,
               'selected_method': selected_method, **seasonal_vector}]).to_csv(
    DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS, index=False
)
diagnostics.drop(columns=['observed', 'predicted']).to_csv(
    OUTPUT / '02_seasonal_poisson_window_diagnostics.csv', index=False
)
selection_rows = [
    {'method': selected_method, 'poisson_nll': search_score, 'selected': True},
]
if RUN_OPTIONAL_GRADIENT:
    selection_rows.append({
        'method': 'optional numerical-gradient L-BFGS-B diagnostic',
        'poisson_nll': gradient_score, 'selected': False,
        'optimizer_success': gradient_result.success,
    })
selection = pd.DataFrame(selection_rows)
selection.to_csv(OUTPUT / '11_seasonal_fit_selection.csv', index=False)
display(selection)
display(fitted_parameter_table(base, seasonal_vector, SEASONAL_FIT_PARAMETER_BOUNDS))
print('Canonical method:', selected_method)
print('Selected amplitude:', seasonal_vector['seasonal_amplitude'])
print('Selected peak week:', seasonal_vector['seasonal_peak_week'])
print('Selected Poisson NLL:', seasonal_score)
if np.isclose(seasonal_vector['seasonal_amplitude'],
              SEASONAL_FIT_PARAMETER_BOUNDS['seasonal_amplitude'][1], atol=0.01):
    print('WARNING: amplitude remains near its upper bound; widen or regularise before interpretation.')

## 3. Like-for-like calibration comparison

The original non-seasonal Poisson vector, the archived seasonal vector fitted to all 117 windows, and the new post-wave seasonal vector are evaluated on the same 86 post-wave rolling windows. This is a calibration-sample sensitivity comparison, not a held-out forecast comparison.

In [ ]:
nonseasonal_score, nonseasonal_diag = evaluate_vector(
    nonseasonal_vector, blocks, inputs, base, overlap_weighted=True,
    objective_metric='poisson_nll'
)
seasonal_score, seasonal_diag = evaluate_vector(
    seasonal_vector, blocks, inputs, base, overlap_weighted=True,
    objective_metric='poisson_nll'
)
all_series_seasonal_score, all_series_seasonal_diag = evaluate_vector(
    all_series_seasonal_vector, blocks, inputs, base, overlap_weighted=True,
    objective_metric='poisson_nll'
)
def flattened_overlap_metrics(diag):
    residuals, weights = [], []
    for row in diag.itertuples():
        block = blocks.loc[blocks.block_id.eq(int(row.block_id))].sort_values('week_in_block')
        residuals.extend(np.asarray(row.predicted) - np.asarray(row.observed))
        weights.extend(1.0 / block.week_coverage.to_numpy(float))
    residuals = np.asarray(residuals, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return (
        float(np.average(np.abs(residuals), weights=weights)),
        float(np.sqrt(np.average(residuals ** 2, weights=weights))),
        float(np.average(residuals, weights=weights)),
    )
def summary_row(label, fitted_windows, score, diag, vector=None):
    weighted_mae, weighted_rmse, weighted_bias = flattened_overlap_metrics(diag)
    return {
        'model': label, 'fitted_windows': fitted_windows,
        'evaluation_windows': blocks.block_id.nunique(), 'poisson_nll': score,
        'weighted_MAE': weighted_mae,
        'weighted_RMSE': weighted_rmse,
        'weighted_bias': weighted_bias,
        'seasonal_amplitude': 0.0 if vector is None else vector['seasonal_amplitude'],
        'seasonal_peak_week': np.nan if vector is None else vector['seasonal_peak_week'],
    }
comparison = pd.DataFrame([
    summary_row('non-seasonal', 117, nonseasonal_score, nonseasonal_diag),
    summary_row('seasonal including 2024 wave', 117, all_series_seasonal_score,
                all_series_seasonal_diag, all_series_seasonal_vector),
    summary_row('seasonal excluding 2024 wave', 86, seasonal_score, seasonal_diag,
                seasonal_vector),
])
comparison.to_csv(OUTPUT / '04_seasonal_vs_nonseasonal_calibration.csv', index=False)
display(comparison)

In [ ]:
def calendar_fit(diagnostics):
    rows = []
    for diagnostic in diagnostics.itertuples():
        block = blocks.loc[blocks.block_id.eq(int(diagnostic.block_id))].sort_values('week_in_block')
        for source_index, date, observed, predicted in zip(
            block.source_week_index, block.date,
            diagnostic.observed, diagnostic.predicted,
        ):
            rows.append({
                'source_week_index': int(source_index), 'date': pd.Timestamp(date),
                'observed_cases': float(observed), 'predicted_cases': float(predicted),
            })
    return pd.DataFrame(rows).groupby(
        ['source_week_index', 'date'], as_index=False
    ).agg(observed_cases=('observed_cases', 'first'),
          median_predicted_cases=('predicted_cases', 'median'),
          p10_predicted_cases=('predicted_cases', lambda values: values.quantile(0.10)),
          p90_predicted_cases=('predicted_cases', lambda values: values.quantile(0.90)),
          contributing_windows=('predicted_cases', 'size'))

nonseasonal_calendar = calendar_fit(nonseasonal_diag)
all_series_seasonal_calendar = calendar_fit(all_series_seasonal_diag)
seasonal_calendar = calendar_fit(seasonal_diag)
calendar_output = pd.concat([
    nonseasonal_calendar.assign(model='non-seasonal'),
    all_series_seasonal_calendar.assign(model='seasonal including 2024 wave'),
    seasonal_calendar.assign(model='seasonal excluding 2024 wave'),
], ignore_index=True)
calendar_output.to_csv(OUTPUT / '07_calendar_predictions.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for ax, calendar, colour, title in [
    (axes[0], nonseasonal_calendar, 'tab:blue', 'Non-seasonal'),
    (axes[1], seasonal_calendar, 'tab:orange', 'Seasonal'),
]:
    ax.fill_between(
        calendar.date, calendar.p10_predicted_cases, calendar.p90_predicted_cases,
        color=colour, alpha=.20, label='rolling-window p10--p90',
    )
    ax.plot(calendar.date, calendar.median_predicted_cases,
            color=colour, lw=1.8, label='rolling-window median')
    ax.plot(cases.date, cases.observed_cases, 'ko-', ms=2.5, lw=1.0,
            label='confirmed London cases')
    ax.set(title=title, xlabel='Week of symptom onset')
    ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Cases per week')
fig.suptitle('Post-wave rolling six-week predictions versus confirmed cases')
fig.tight_layout()
fig.savefig(OUTPUT / '06_calendar_fit_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for ax, calendar, colour, title in [
    (axes[0], all_series_seasonal_calendar, 'tab:purple',
     'Seasonal vector fitted including the 2024 wave'),
    (axes[1], seasonal_calendar, 'tab:orange',
     'Seasonal vector fitted excluding the 2024 wave'),
]:
    ax.fill_between(
        calendar.date, calendar.p10_predicted_cases, calendar.p90_predicted_cases,
        color=colour, alpha=.20, label='rolling-window p10--p90',
    )
    ax.plot(calendar.date, calendar.median_predicted_cases, color=colour,
            lw=1.8, label='rolling-window median')
    ax.plot(cases.date, cases.observed_cases, 'ko-', ms=2.5, lw=1.0,
            label='confirmed London cases')
    ax.set(title=title, xlabel='Week of symptom onset')
    ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Cases per week')
fig.suptitle('Sensitivity to excluding the exceptional 2024 wave from seasonal fitting')
fig.tight_layout()
fig.savefig(OUTPUT / '08_seasonal_sample_sensitivity.png', dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(trials.index, trials.objective, s=13, alpha=.55)
axes[0].axhline(seasonal_score, color='tab:red', ls='--')
axes[0].set(title='Seasonal optimisation history', xlabel='Saved candidate',
            ylabel='Overlap-weighted Poisson NLL')
best_so_far = trials.objective.cummin()
axes[1].plot(best_so_far.to_numpy(), color='tab:green')
axes[1].set(title='Best score found', xlabel='Saved candidate', ylabel='Poisson NLL')
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(OUTPUT / '05_seasonal_optimization.png', dpi=180, bbox_inches='tight')
plt.show()

## Interpretation and next run

- The selected amplitude and phase are estimates conditional on the model, bounds, observation loss and London series; they are not clinical constants or confidence intervals.
- A lower in-sample Poisson loss is expected from a more flexible model and is not sufficient for adoption.
- Rerun `London_Seasonal_Complete_Rolling_Audit.ipynb` using this saved parameter file, with 500--1,000 paths for final probability scores.
- For a genuinely out-of-sample claim, fit the seasonal vector using only the designated training period and freeze it before evaluating later origins.
- Report the selected numerical values and performance comparison in Results; Methods should report the equation, bounds and fitting procedure.